# affect_aif Paper Reproduction Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/har5h1l/affect_aif/blob/main/notebooks/reproduce.ipynb)

Run the paper reproduction configs, regenerate compact analysis artifacts, plot paper readouts, and export a clean `results/` folder. This notebook is Colab-compatible and local-Jupyter-compatible.

Top-to-bottom execution runs the experiments first into `outputs/paper_full/`, materializes those fresh outputs into the canonical gitignored `results/paper/*/raw/` layout, and then analyzes the materialized files.


## 1. Bootstrap Runtime

In Colab this clones the repo into `/content/affect_aif`. Locally, launch Jupyter from the repo root. GPU is optional; use a GPU runtime if available.


In [ ]:
from pathlib import Path
import json
import os
import platform
import shutil
import subprocess
import sys
import textwrap

IN_COLAB = Path("/content").exists()
REPO_URL = os.environ.get("AFFECT_AIF_REPO_URL", "https://github.com/har5h1l/affect_aif.git")
BRANCH = os.environ.get("AFFECT_AIF_BRANCH", "main")

if IN_COLAB:
    os.chdir("/content")
    if not Path("affect_aif").exists():
        subprocess.run(["git", "clone", "--branch", BRANCH, REPO_URL, "affect_aif"], check=True)
    os.chdir("/content/affect_aif")

ROOT = Path.cwd()
if not (ROOT / "scripts" / "experiment" / "run.py").exists():
    raise RuntimeError("Run this notebook from the affect_aif repo root, or use the Colab bootstrap clone.")

print("Repo root:", ROOT)
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())


## 2. Install Dependencies

Colab runtimes start clean, so install the project into the runtime. Local users with an existing editable install can set `INSTALL_DEPS = False`.


In [ ]:
INSTALL_DEPS = True

if INSTALL_DEPS:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)
    print("Installed affect_aif in editable mode for this runtime.")
else:
    print("Skipped dependency install.")


## 3. Check Runtime Device

This reports whether JAX sees an accelerator. The experiments should run on CPU too; the GPU check is for transparency and Colab readiness.


In [ ]:
try:
    import jax
    print("JAX devices:", jax.devices())
except Exception as exc:
    print("JAX device check unavailable:", exc)

if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi"], check=False)
else:
    print("No NVIDIA GPU visible. This notebook also runs on CPU; GPU is optional for these trust-task sweeps.")


## 4. Paper Configs and Execution Flags

The notebook always starts with a dry-run manifest so you can see the expanded workload. `RUN_EXPERIMENTS` then launches the paper configs into `outputs/paper_full/`. `MATERIALIZE_RESULTS` copies those fresh outputs into the canonical gitignored `results/paper/*/raw/` layout before analysis.


In [ ]:
RUN_EXPERIMENTS = True
MATERIALIZE_RESULTS = True
RUN_ANALYSIS = True
EXPORT_TO_DRIVE = False

OUTPUT_ROOT = Path("outputs")
PAPER_BATCH = "paper_full"
PAPER_OUT = OUTPUT_ROOT / PAPER_BATCH
MANUSCRIPT_DIR = Path("docs/manuscript")
RESULTS_ROOT = Path("results")

PAPER_CONFIGS = {
    "model_fitness": "configs/paper/h1_model_fitness/reliability_vs_reward_confirm.toml",
    "betrayal_adaptation": "configs/paper/h5_betrayal/betrayal_reallocation_confirm.toml",
    "alpha_sweep": "configs/paper/alpha_sweep.toml",
    "prior_factorial": "configs/paper/prior_factorial.toml",
    "forgiveness": "configs/paper/forgiveness.toml",
    "mixed_volatility": "configs/paper/mixed_volatility.toml",
}

for name, config in PAPER_CONFIGS.items():
    if not Path(config).exists():
        raise FileNotFoundError(f"Missing paper config for {name}: {config}")

CANONICAL_RAW = {
    "model_fitness": Path("results/paper/model_fitness/raw/h1/reliability_vs_reward_confirm/results.csv"),
    "betrayal_adaptation": Path("results/paper/betrayal_adaptation/raw/h5/betrayal_reallocation_confirm/results.csv"),
    "alpha_sweep": Path("results/paper/alpha_sweep/raw/results.csv"),
    "prior_factorial": Path("results/paper/prior_factorial/raw/results.csv"),
    "forgiveness": Path("results/paper/forgiveness/raw/results.csv"),
    "mixed_volatility": Path("results/paper/mixed_volatility/raw/results.csv"),
}

print("Paper configs:")
for name, config in PAPER_CONFIGS.items():
    print(f"- {name}: {config}")


## 5. Dry-Run Manifest

The dry run validates configs and reports expanded run counts before any expensive work starts.


In [ ]:
dry_cmd = [sys.executable, "scripts/experiment/run.py"]
for config in PAPER_CONFIGS.values():
    dry_cmd.extend(["--config", config])
dry_cmd.extend(["--output-dir", str(OUTPUT_ROOT), "--batch-name", PAPER_BATCH + "_dry", "--workers", "1", "--dry-run"])

print("Command:", " ".join(dry_cmd))
subprocess.run(dry_cmd, check=True)
manifest_path = OUTPUT_ROOT / (PAPER_BATCH + "_dry") / "manifest.json"
manifest = json.loads(manifest_path.read_text())

import pandas as pd
manifest_frame = pd.DataFrame(manifest["configs"])
display(manifest_frame[["hypothesis_id", "experiment_id", "rounds", "replications", "expanded_runs", "path"]])
print("Total expanded runs:", int(manifest_frame["expanded_runs"].sum()))


## 6. Full Paper Run

This cell runs the paper configs into `outputs/paper_full/`. Set `RUN_EXPERIMENTS = False` only when you intentionally want to inspect the dry-run manifest without reproducing the paper outputs.


In [ ]:
full_cmd = [sys.executable, "scripts/experiment/run.py"]
for config in PAPER_CONFIGS.values():
    full_cmd.extend(["--config", config])
full_cmd.extend(["--output-dir", str(OUTPUT_ROOT), "--batch-name", PAPER_BATCH, "--workers", "1"])

print("Command:", " ".join(full_cmd))
if RUN_EXPERIMENTS:
    subprocess.run(full_cmd, check=True)
else:
    print("Skipped full run. Set RUN_EXPERIMENTS = True to reproduce paper outputs.")


## 7. Materialize Canonical Results

After the full run, this copies final `results.csv` files from `outputs/paper_full/` into the canonical result layout. If an old canonical raw folder exists, it is replaced with the freshly reproduced output so later cells do not accidentally analyze stale local data.


In [ ]:
def copy_dir(src: Path, dst: Path):
    if not src.exists():
        raise FileNotFoundError(src)
    dst.parent.mkdir(parents=True, exist_ok=True)
    if dst.exists():
        shutil.rmtree(dst)
    shutil.copytree(src, dst)


def combine_results(paths, output_path: Path):
    frames = []
    for path in paths:
        frame = pd.read_csv(path, low_memory=False)
        frames.append(frame)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    pd.concat(frames, ignore_index=True).to_csv(output_path, index=False)


def materialize_from_batch(batch_dir: Path):
    mappings = [
        (batch_dir / "h1" / "reliability_vs_reward_confirm", Path("results/paper/model_fitness/raw/h1/reliability_vs_reward_confirm")),
        (batch_dir / "h5" / "betrayal_reallocation_confirm", Path("results/paper/betrayal_adaptation/raw/h5/betrayal_reallocation_confirm")),
        (batch_dir / "exp_c" / "forgiveness", Path("results/paper/forgiveness/raw/forgiveness")),
        (batch_dir / "exp_d" / "mixed_volatility", Path("results/paper/mixed_volatility/raw/mixed_volatility")),
    ]
    for src, dst in mappings:
        if src.exists():
            copy_dir(src, dst)

    suite_mappings = {
        "alpha_sweep": (batch_dir / "exp_a", Path("results/paper/alpha_sweep/raw")),
        "prior_factorial": (batch_dir / "exp_b", Path("results/paper/prior_factorial/raw")),
    }
    for _, (src_root, dst_root) in suite_mappings.items():
        if not src_root.exists():
            continue
        child_results = []
        for child in sorted(src_root.iterdir()):
            if not child.is_dir():
                continue
            dst = dst_root / child.name
            copy_dir(child, dst)
            result_path = dst / "results.csv"
            if result_path.exists():
                child_results.append(result_path)
        if child_results:
            combine_results(child_results, dst_root / "results.csv")

    if Path("results/paper/forgiveness/raw/forgiveness/results.csv").exists():
        shutil.copy2(Path("results/paper/forgiveness/raw/forgiveness/results.csv"), Path("results/paper/forgiveness/raw/results.csv"))
    if Path("results/paper/mixed_volatility/raw/mixed_volatility/results.csv").exists():
        shutil.copy2(Path("results/paper/mixed_volatility/raw/mixed_volatility/results.csv"), Path("results/paper/mixed_volatility/raw/results.csv"))

if MATERIALIZE_RESULTS:
    if not PAPER_OUT.exists():
        raise FileNotFoundError(f"Missing fresh paper run output: {PAPER_OUT}")
    materialize_from_batch(PAPER_OUT)
else:
    print("Skipped materialization. Set MATERIALIZE_RESULTS = True to analyze freshly reproduced outputs.")

print("Canonical raw availability:")
for name, path in CANONICAL_RAW.items():
    print(f"- {name}: {path.exists()} {path}")


## 8. Regenerate Analysis Artifacts

This cell analyzes the canonical raw files produced by the preceding run/materialization step. Missing raw files are treated as an error so the notebook does not silently fall back to an incomplete or stale result packet.


In [ ]:
def run_required(cmd, required_path: Path):
    if not required_path.exists():
        raise FileNotFoundError(f"Missing raw result required for analysis: {required_path}")
    print("Running:", " ".join(map(str, cmd)))
    subprocess.run([str(x) for x in cmd], check=True)

if RUN_ANALYSIS:
    run_required([
        sys.executable, "scripts/analysis/analyze.py",
        "--results", CANONICAL_RAW["model_fitness"],
        "--output-dir", CANONICAL_RAW["model_fitness"].parent / "analysis",
        "--config", PAPER_CONFIGS["model_fitness"],
    ], CANONICAL_RAW["model_fitness"])

    run_required([
        sys.executable, "scripts/analysis/analyze.py",
        "--results", CANONICAL_RAW["betrayal_adaptation"],
        "--output-dir", CANONICAL_RAW["betrayal_adaptation"].parent / "analysis",
        "--config", PAPER_CONFIGS["betrayal_adaptation"],
    ], CANONICAL_RAW["betrayal_adaptation"])

    for experiment in ["alpha_sweep", "prior_factorial", "forgiveness", "mixed_volatility"]:
        raw_root = CANONICAL_RAW[experiment].parent
        if not CANONICAL_RAW[experiment].exists():
            raise FileNotFoundError(f"Missing raw result required for analysis: {CANONICAL_RAW[experiment]}")
        cmd = [
            sys.executable,
            "scripts/analysis/phenotype_artifacts.py",
            experiment,
            "--results-root",
            raw_root,
            "--paper-dir",
            MANUSCRIPT_DIR,
        ]
        print("Running:", " ".join(map(str, cmd)))
        subprocess.run([str(x) for x in cmd], check=True)
else:
    print("Analysis skipped.")


## 9. Verify Result Packet Cleanliness

The local `results/` folder should be safe to copy to Drive: no checkpoints, no partial CSVs, raw files under semantic paper folders, and manifests/summaries at the top of each result folder.


In [ ]:
partial_files = sorted(Path("results").glob("**/results_partial.csv")) + sorted(Path("results").glob("**/checkpoint_manifest.json"))
if partial_files:
    raise RuntimeError("Partial/checkpoint files remain:\n" + "\n".join(map(str, partial_files[:20])))

raw_count = sum(1 for _ in Path("results/paper").glob("**/raw/**/results.csv")) if Path("results/paper").exists() else 0
print("Paper raw results.csv count:", raw_count)

for name, path in CANONICAL_RAW.items():
    print(f"{name:22s}", "OK" if path.exists() else "missing", path)

if shutil.which("git"):
    probe = Path("results/paper/alpha_sweep/raw/results.csv")
    if probe.exists():
        check = subprocess.run(["git", "check-ignore", "-q", str(probe)], check=False)
        print("Raw result gitignored:", check.returncode == 0)


## 10. Load Paper Summaries

This cell loads compact summary and metrics tables when present. The notebook uses these for plots and interpretation snippets.


In [ ]:
def load_csv(path):
    path = Path(path)
    if path.exists():
        return pd.read_csv(path)
    print("Missing:", path)
    return pd.DataFrame()

paper_tables = {
    "model_fitness_final": load_csv("results/paper/model_fitness/source_tables/final_round_summary.csv"),
    "model_fitness_evidence": load_csv("results/paper/model_fitness/source_tables/h1_evidence_effect_summary.csv"),
    "betrayal_evidence": load_csv("results/paper/betrayal_adaptation/source_tables/h5_evidence_effect_summary.csv"),
    "alpha_sweep": load_csv("results/paper/alpha_sweep/metrics.csv"),
    "prior_factorial": load_csv("results/paper/prior_factorial/metrics.csv"),
    "forgiveness": load_csv("results/paper/forgiveness/metrics.csv"),
    "mixed_volatility": load_csv("results/paper/mixed_volatility/metrics.csv"),
}

for name, table in paper_tables.items():
    print(name, table.shape)


## 11. Core Mechanism Plots

These plots summarize paper-level mechanism readouts when the corresponding tables are available.


In [ ]:
figures = []

mf = paper_tables["model_fitness_final"]
if not mf.empty and {"variant_id", "cumulative_payoff"} <= set(mf.columns):
    mf_plot = mf.groupby("variant_id", as_index=False)["cumulative_payoff"].mean()
    fig, ax = plt.subplots(figsize=(6, 3.5))
    ax.bar(mf_plot["variant_id"], mf_plot["cumulative_payoff"])
    ax.set(title="H1 model-fitness: mean cumulative payoff", xlabel="Variant", ylabel="Cumulative payoff")
    ax.tick_params(axis="x", rotation=30)
    plt.tight_layout(); plt.show()

h5 = paper_tables["betrayal_evidence"]
if not h5.empty:
    numeric_cols = [c for c in h5.columns if pd.api.types.is_numeric_dtype(h5[c])]
    display(h5.head())
    if "variant_id" in h5.columns and numeric_cols:
        metric = numeric_cols[0]
        fig, ax = plt.subplots(figsize=(6, 3.5))
        h5_plot = h5.groupby("variant_id", as_index=False)[metric].mean()
        ax.bar(h5_plot["variant_id"], h5_plot[metric])
        ax.set(title=f"H5 betrayal adaptation: {metric}", xlabel="Variant", ylabel=metric)
        ax.tick_params(axis="x", rotation=30)
        plt.tight_layout(); plt.show()


## 12. Phenotype Plots

These plots cover Exp A-D from the compact `metrics.csv` files. They are descriptive computational-profile readouts, not clinical validation.


In [ ]:
def plot_metric(table, title, metric, by="variant_id"):
    if table.empty or metric not in table.columns or by not in table.columns:
        print("Skipping", title, "missing", metric)
        return
    plot = table.groupby(by, as_index=False)[metric].mean()
    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.bar(plot[by].astype(str), plot[metric])
    ax.set(title=title, xlabel=by, ylabel=metric)
    ax.tick_params(axis="x", rotation=35)
    plt.tight_layout(); plt.show()

plot_metric(paper_tables["alpha_sweep"], "Exp A: alpha vs entropy late", "entropy_late")
plot_metric(paper_tables["prior_factorial"], "Exp B: trust asymmetry", "trust_asymmetry")
plot_metric(paper_tables["forgiveness"], "Exp C: reengagement rate", "reengagement_rate")
plot_metric(paper_tables["mixed_volatility"], "Exp D: discrimination index", "discrimination_index")
plot_metric(paper_tables["mixed_volatility"], "Exp D: false positive rate", "false_positive_rate")


## 13. Interpretation Summary

This cell prints a compact, table-derived interpretation scaffold. Use it as a reproducibility readout, not as a substitute for the manuscript text.


In [ ]:
def best_variant(table, metric, higher=True):
    if table.empty or metric not in table.columns or "variant_id" not in table.columns:
        return "unavailable"
    grouped = table.groupby("variant_id", as_index=False)[metric].mean().dropna()
    if grouped.empty:
        return "unavailable"
    row = grouped.sort_values(metric, ascending=not higher).iloc[0]
    return f"{row.variant_id} ({metric}={row[metric]:.3g})"

print("Paper reproduction interpretation scaffold")
print("- H1 model fitness: inspect model_fitness source tables for surprise-over-reward tracking and payoff boundary.")
print("- H5 betrayal adaptation: inspect betrayal source tables for entropy/accuracy/payoff under abrupt partner change.")
print("- Exp A alpha sweep: lowest late entropy:", best_variant(paper_tables["alpha_sweep"], "entropy_late", higher=False))
print("- Exp B prior factorial: strongest trust asymmetry:", best_variant(paper_tables["prior_factorial"], "trust_asymmetry", higher=True))
print("- Exp C forgiveness: strongest reengagement:", best_variant(paper_tables["forgiveness"], "reengagement_rate", higher=True))
print("- Exp D mixed volatility: strongest discrimination:", best_variant(paper_tables["mixed_volatility"], "discrimination_index", higher=True))
print("- Exp D mixed volatility: lowest false-positive rate:", best_variant(paper_tables["mixed_volatility"], "false_positive_rate", higher=False))
print("\nGuardrail: these are computational profile readouts. Do not phrase them as human, clinical, or payoff-superiority claims without manuscript review.")


## 14. Export Clean Results To Google Drive

After full reproduction and analysis, this copies the local `results/` folder to Drive. The folder is designed to be shareable: compact summaries are tracked; raw paper data lives under `results/paper/*/raw`; checkpoints and partial CSVs should be absent.


In [ ]:
if EXPORT_TO_DRIVE:
    if not IN_COLAB:
        raise RuntimeError("Google Drive export is only available in Colab.")
    from google.colab import drive
    drive.mount("/content/drive")
    drive_dest = Path("/content/drive/MyDrive/affect_aif_results")
    drive_dest.mkdir(parents=True, exist_ok=True)
    subprocess.run(["rsync", "-a", "results/", str(drive_dest) + "/"], check=True)
    print("Copied results/ to", drive_dest)
else:
    print("Drive export skipped. Set EXPORT_TO_DRIVE = True to copy results/ to Google Drive in Colab.")
